# Part 3 — μP (Maximal Update Parameterization): Tasks 3.1–3.3

**CS-GY 6923 · NYU Tandon · Spring 2026**

This notebook covers:
- **Task 3.1** — Conceptual overview of μP
- **Task 3.2** — Implementation: what changed vs. Standard Parameterization (SP)
- **Task 3.3** — μP LR sweep on the Tiny model + comparison with SP sweep

Key promise of μP: **the optimal learning rate found on the Tiny model transfers to all larger widths without retuning.**

In [ ]:
import os, sys, json, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Navigate to project root
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), 'ML-Final-Project'))
if not os.path.exists('configs'):
    os.chdir('..')
print('Working directory:', os.getcwd())

---
## Task 3.1 — μP Conceptually

### The Problem with Standard Parameterization (SP)

Under SP (standard PyTorch `nn.Linear` + `AdamW`), the optimal learning rate **shrinks as model width increases**. This means:
- You must re-run an expensive LR sweep for every new model size.
- LR tuning at small scale gives misleading results for large models.

Why does this happen? In a width-$d$ hidden layer, the forward pass computes $\mathbf{h} = W\mathbf{x}$ where $W \in \mathbb{R}^{d \times d}$. With standard init ($\sigma \propto 1/\sqrt{d}$) and a fixed learning rate $\eta$, the parameter update $\Delta W \sim \eta / \sqrt{d}$ causes the **output change** $\|\Delta \mathbf{h}\| \propto \eta \cdot \sqrt{d} \cdot \|\mathbf{x}\|$ to **grow with width** — destabilizing training.

### What μP Does (Yang et al., Tensor Programs V, 2022)

μP re-scales **initialization** and **per-layer learning rate multipliers** so that:

1. Feature updates $\|\Delta \mathbf{h}\|$ remain $O(1)$ regardless of width.
2. The optimal LR at small width **transfers exactly** to all larger widths.

### Three Key Changes for Transformers

| Component | SP | μP | Why |
|-----------|----|----|-----|
| Attention logit scale | $1/\sqrt{d_\text{head}}$ | $1/d_\text{head}$ | Keeps attention entropy stable across widths |
| Output (`lm_head`) LR | $\eta$ | $\eta \cdot (d_\text{base}/d_\text{model})$ | Readout fan-in scales with width; LR must shrink |
| Hidden layers LR | $\eta$ | $\eta \cdot (d_\text{base}/d_\text{model})$ | MuAdamW applies this automatically via `infshape` |

No weight tying between input embedding and output head — they require different LR multipliers.

---
## Task 3.2 — Implementation

All μP code lives in `scripts/mup_model.py` and `scripts/mup_train.py`.
The changes relative to `model.py` and `train.py` are minimal and surgical.

In [ ]:
# Verify mup is installed
from mup import MuReadout, set_base_shapes, MuAdamW
print('mup package: OK')

from scripts.model    import ModelConfig
from scripts.mup_model import MupGPT, build_mup_model
print('MupGPT imports: OK')

### Change 1 — Attention Scale: `1/head_dim` instead of `1/√head_dim`

```python
# SP  (model.py)
scale = 1.0 / math.sqrt(self.head_dim)   # shrinks as width grows

# μP  (mup_model.py)
self._mup_scale = 1.0 / self.head_dim    # linear in head_dim
y = F.scaled_dot_product_attention(q, k, v, scale=self._mup_scale, ...)
```

### Change 2 — Output Layer: `MuReadout` instead of `nn.Linear`

```python
# SP  (model.py)
self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
self.transformer.wte.weight = self.lm_head.weight  # weight tying

# μP  (mup_model.py)
self.lm_head = MuReadout(config.d_model, config.vocab_size, bias=False)
# No weight tying — MuReadout gets a different LR multiplier than wte
```

### Change 3 — `build_mup_model()` registers base shapes

```python
def build_mup_model(config: ModelConfig) -> MupGPT:
    base_cfg  = ModelConfig(d_model=64,  n_layers=config.n_layers, ...)  # same depth, 64-wide
    delta_cfg = ModelConfig(d_model=128, n_layers=config.n_layers, ...)  # same depth, 128-wide
    base_model  = MupGPT(base_cfg)
    delta_model = MupGPT(delta_cfg)
    model       = MupGPT(config)
    set_base_shapes(model, base_model, delta=delta_model)   # registers μP infshapes
    return model
```

### Change 4 — `MuAdamW` instead of `AdamW`

```python
# SP
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, ...)

# μP
optimizer = MuAdamW(model.parameters(), lr=lr, ...)  # reads infshapes → applies lr multipliers
```

In [ ]:
# Show parameter counts for all 5 μP model sizes
configs = [
    ('tiny',   'configs/tiny.json'),
    ('small',  'configs/small.json'),
    ('medium', 'configs/medium.json'),
    ('large',  'configs/large.json'),
    ('xl',     'configs/xl.json'),
]

print(f"{'Name':<8} {'d_model':>8} {'n_layers':>9} {'n_heads':>8} {'SP params':>12} {'μP params':>12} {'Note'}")
print('-' * 75)

from scripts.model import GPT
for name, path in configs:
    with open(path) as f:
        cfg_dict = json.load(f)
    cfg = ModelConfig.from_dict(cfg_dict)
    sp_params  = GPT(cfg).count_parameters()
    mup_params = build_mup_model(cfg).count_parameters()
    delta = mup_params - sp_params
    print(f"{name:<8} {cfg.d_model:>8} {cfg.n_layers:>9} {cfg.n_heads:>8} "
          f"{sp_params:>12,} {mup_params:>12,}  +{delta:,} (no weight tying)")

The μP models have slightly more parameters than SP because weight tying is removed (embedding and lm_head are separate). The extra parameters equal `vocab_size × d_model` for each model.

---
## Task 3.3 — μP Learning Rate Sweep

Same protocol as Task 2.2 (SP sweep):
- 7 LRs on a log scale: `[1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2]`
- Tiny model only, 20% of one epoch (310 steps) per trial
- Best LR used for **all** model sizes without retuning — that is the μP guarantee

Run via: `python scripts/mup_lr_sweep.py --sweep_fraction 0.2`

In [ ]:
# Load both sweep results
with open('outputs/results/lr_sweep_results.json')     as f: sp_raw  = json.load(f)
with open('outputs/results/mup_lr_sweep_results.json') as f: mup_raw = json.load(f)

sp_results  = {float(k): v for k, v in sp_raw.items()}
mup_results = {float(k): v for k, v in mup_raw.items()}

lrs = sorted(sp_results.keys())

sp_best_lr  = min(sp_results,  key=sp_results.get)
mup_best_lr = min(mup_results, key=mup_results.get)

print('SP  LR Sweep Results (Task 2.2):')
print(f"  {'LR':>10}  {'Val Loss':>10}")
print('  ' + '-'*24)
for lr in lrs:
    marker = '  ← BEST' if lr == sp_best_lr else ''
    print(f"  {lr:>10.2e}  {sp_results[lr]:>10.4f}{marker}")

print()
print('μP  LR Sweep Results (Task 3.3):')
print(f"  {'LR':>10}  {'Val Loss':>10}")
print('  ' + '-'*24)
for lr in lrs:
    marker = '  ← BEST' if lr == mup_best_lr else ''
    print(f"  {lr:>10.2e}  {mup_results[lr]:>10.4f}{marker}")

lr_ratio = mup_best_lr / sp_best_lr
print(f'\nμP best LR / SP best LR = {mup_best_lr:.1e} / {sp_best_lr:.1e} = {lr_ratio:.1f}×')

### Side-by-Side Plot: SP vs μP LR Sweep

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, data, best, label, color, marker in [
    (axes[0], sp_results,  sp_best_lr,  'SP — Standard',        'steelblue',  'o'),
    (axes[1], mup_results, mup_best_lr, 'μP — MuP (Task 3.3)',  'darkorchid', 's'),
]:
    x = sorted(data.keys())
    y = [data[lr] for lr in x]
    ax.semilogx(x, y, f'{marker}-', color=color, linewidth=2, markersize=8, label=label)
    ax.axvline(best, color='tomato', linestyle='--', linewidth=1.8,
               label=f'Best LR = {best:.1e}  (val={data[best]:.4f})')
    for lr, v in zip(x, y):
        ax.annotate(f'{v:.3f}', (lr, v), textcoords='offset points',
                    xytext=(0, 9), ha='center', fontsize=8)
    ax.set_xlabel('Learning Rate (log scale)', fontsize=11)
    ax.set_ylabel('Validation Loss', fontsize=11)
    ax.set_title(f'{label}\nTiny Model — 20% epoch (310 steps)', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, which='both', alpha=0.3)

fig.suptitle('Task 3.3 — LR Sweep Comparison: SP vs μP', fontsize=13, fontweight='bold')
fig.tight_layout()
os.makedirs('outputs/plots', exist_ok=True)
fig.savefig('outputs/plots/lr_sweep_comparison.png', dpi=150)
plt.show()
print('Saved: outputs/plots/lr_sweep_comparison.png')

### Overlay: Both Sweeps on the Same Axes

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for data, best, label, color, marker in [
    (sp_results,  sp_best_lr,  'SP (Standard Param.)',       'steelblue',  'o'),
    (mup_results, mup_best_lr, 'μP (Maximal Update Param.)', 'darkorchid', 's'),
]:
    x = sorted(data.keys())
    y = [data[lr] for lr in x]
    ax.semilogx(x, y, f'{marker}-', color=color, linewidth=2, markersize=8, label=label)
    ax.axvline(best, color=color, linestyle=':', linewidth=1.5, alpha=0.7)
    ax.annotate(f'Best\n{best:.1e}', (best, ax.get_ylim()[1] if ax.get_ylim()[1] < 10 else 6.0),
                textcoords='offset points', xytext=(5, -20), fontsize=9, color=color)

ax.set_xlabel('Learning Rate (log scale)', fontsize=12)
ax.set_ylabel('Validation Loss (20% epoch)', fontsize=12)
ax.set_title('SP vs μP — LR Sensitivity on Tiny Model\n'
             'μP curve continues improving at higher LR; SP diverges at 1e-2', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
fig.savefig('outputs/plots/lr_sweep_overlay.png', dpi=150)
plt.show()
print('Saved: outputs/plots/lr_sweep_overlay.png')

---
## Findings & Analysis

### Numerical Results

| LR | SP val loss | μP val loss | Δ (μP − SP) |
|----|-------------|-------------|-------------|
| 1e-5 | 6.0661 | 6.4869 | +0.421 |
| 3e-5 | 5.4399 | 6.1460 | +0.706 |
| 1e-4 | 4.2819 | 5.4299 | +1.148 |
| 3e-4 | 3.0435 | 3.9484 | +0.905 |
| 1e-3 | 2.6896 | 2.8408 | +0.151 |
| **3e-3** | **2.3214 ← SP best** | 2.4541 | +0.133 |
| 1e-2 | 2.3567 ↑ (SP peaks)  | **2.2587 ← μP best** | **−0.098** |

### Key Observations

**1. Different optimal LRs:**
- SP best LR = **3e-3**
- μP best LR = **1e-2** (3.3× higher)

Under SP, training diverges or degrades at 1e-2 (val loss rises from 2.32 → 2.36). Under μP, 1e-2 is still improving — no divergence.

**2. Why μP tolerates higher LR:**
The `1/head_dim` attention scaling (vs `1/√head_dim`) produces smaller raw logits, which means attention outputs have smaller magnitude. Combined with MuAdamW's per-layer LR multipliers, the effective per-parameter update magnitude stays bounded even at LR=1e-2.

**3. μP is worse at low LR, better at high LR:**
At LRs 1e-5 through 1e-3, μP is strictly *worse* than SP on the Tiny model. This is expected — μP is not optimized for small models; it's optimized so that the LR that works at small scale **also works at large scale**.

**4. Best LR is at the edge of the sweep (1e-2):**
The μP sweep is monotonically decreasing across all 7 LRs tested. The true optimal might be slightly above 1e-2. For this study, 1e-2 is used as the transferred LR for Tasks 3.4–3.5.

**5. LR transfer claim:**
Under μP, the LR found on the Tiny model (1e-2) should transfer to Small, Medium, Large, and XL **without retuning**. We verify this in Task 3.4 by training all sizes at LR=1e-2 and checking that larger models don't diverge.

In [ ]:
# Summary bar chart: best val loss comparison at the optimal LR for each method
labels = [f'{lr:.0e}' for lr in lrs]
sp_vals  = [sp_results[lr]  for lr in lrs]
mup_vals = [mup_results[lr] for lr in lrs]

x = np.arange(len(lrs))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_sp  = ax.bar(x - w/2, sp_vals,  w, label='SP',  color='steelblue',  alpha=0.8)
bars_mup = ax.bar(x + w/2, mup_vals, w, label='μP',  color='darkorchid', alpha=0.8)

# Highlight best bars
sp_best_idx  = sp_vals.index(min(sp_vals))
mup_best_idx = mup_vals.index(min(mup_vals))
bars_sp[sp_best_idx].set_edgecolor('gold');   bars_sp[sp_best_idx].set_linewidth(2.5)
bars_mup[mup_best_idx].set_edgecolor('gold'); bars_mup[mup_best_idx].set_linewidth(2.5)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel('Learning Rate', fontsize=12)
ax.set_ylabel('Validation Loss (20% epoch)', fontsize=12)
ax.set_title('SP vs μP Validation Loss at Each LR — Tiny Model\n'
             'Gold border = best per method', fontsize=11)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('outputs/plots/lr_sweep_bar_comparison.png', dpi=150)
plt.show()
print('Saved: outputs/plots/lr_sweep_bar_comparison.png')

---
## Show Saved μP Sweep Plot

In [ ]:
from PIL import Image
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, path, title in [
    (axes[0], 'outputs/plots/lr_sweep.png',     'SP LR Sweep (Task 2.2)'),
    (axes[1], 'outputs/plots/mup_lr_sweep.png', 'μP LR Sweep (Task 3.3)'),
]:
    if os.path.exists(path):
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(title, fontsize=12)
    ax.axis('off')
fig.tight_layout()
plt.show()

---
## Next Steps — Task 3.4

Train all 5 model sizes with the transferred μP LR of **1e-2** (no retuning per size):

```bash
# One-shot: reads best_lr from mup_lr_sweep_results.json automatically
python scripts/mup_train_all.py --auto_lr

# Or explicitly:
python scripts/mup_train_all.py --best_lr 1e-2
```

After training, compare SP vs μP scaling curves:
```bash
python scripts/compare_scaling.py
```

Expected outputs:
- `outputs/plots/sp_vs_mup_scaling.png` — overlay log-log scaling plot
- `outputs/plots/lr_sweep_comparison.png` — side-by-side LR sweeps
- `outputs/results/comparison_results.json` — dual power-law fits + extrapolation to ~880M params